# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
PROJECT_ROOT = "/Users/tomnguyen/Library/CloudStorage/OneDrive-TheUniversityofSydney(Students)/uni/2026/deep_learning/Assignment1_2026/"

In [2]:
# Install Python dependencies (run once per session)
!pip install -r '/Users/tomnguyen/Library/CloudStorage/OneDrive-TheUniversityofSydney(Students)/uni/2026/deep_learning/Assignment1_2026/requirements.txt' -q
!python -m spacy download en

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.0 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [3]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /Users/tomnguyen/Library/CloudStorage/OneDrive-TheUniversityofSydney(Students)/uni/2026/deep_learning/Assignment1_2026


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [13]:
#from Tools.download import download_mini

#download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
# from Tools.preproc import preprocess

# preprocess(
#     train_file="_data/squad/train-mini.json",
#     dev_file="_data/squad/dev-v1.1.json",
#     glove_word_file="_data/glove/glove.mini.txt",
#     target_dir="_data",
#     para_limit=400,
#     ques_limit=50,
# )

Generating train examples…


100%|██████████| 150/150 [00:02<00:00, 54.18it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:00<00:00, 50.67it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:04<00:00, 6891.60it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:01<00:00, 6553.17it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data/train.npz',
 'dev_record_file': '_data/dev.npz',
 'word_emb_file': '_data/word_emb.json',
 'char_emb_file': '_data/char_emb.json',
 'train_eval_file': '_data/train_eval.json',
 'dev_eval_file': '_data/dev_eval.json',
 'word2idx_file': '_data/word2idx.json',
 'char2idx_file': '_data/char2idx.json',
 'dev_meta_file': '_data/dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [15]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    #num_steps  = 60000,
    #batch_size = 8,
    #seed       = 42,

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 6000,
    batch_size = 8,
    seed = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "cosine",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

 30%|███       | 61/200 [04:55<11:14,  4.85s/it]


KeyboardInterrupt: 

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [28:16<00:00,  1.30s/it]


TEST  loss 44.620523  F1 7.542209  EM 0.086001
F1: 7.5422  |  EM: 0.0860  |  Loss: 44.620523


## Experiment

In [4]:
from TrainTools.train import train

common = dict(
    train_npz="_data/train.npz",
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    train_eval_json="_data/train_eval.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    num_steps=6000,
    batch_size=8,
    seed=42,
    optimizer_name="adam",
    loss_name="qa_nll",
    learning_rate=1e-3,
    log_every_steps=10,
    track_cq_stats=True,
    save_step_metrics=True,
)

r_lambda = train(**common, scheduler_name="lambda", step_metrics_file="exp2_lambda_seed42.json")
r_cosine = train(**common, scheduler_name="cosine", step_metrics_file="exp2_cosine_seed42.json")
r_none   = train(**common, scheduler_name="none",   step_metrics_file="exp2_none_seed42.json")

  3%|▎         | 6/200 [00:32<17:33,  5.43s/it]


KeyboardInterrupt: 